# SSD Model Training


In [ ]:
!pip install torch torchvision albumentations tqdm pycocotools

In [ ]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torch.optim.lr_scheduler import MultiStepLR
from torch.cuda.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

In [ ]:
# --- Config ---
PROJECT_DIR = Path('/content/Recicling_Project')
SSD_DIR = PROJECT_DIR / 'ssd_implementation' / 'dataset_ssd'

CLASSES = ['background', 'amarillo', 'azul', 'verde', 'marron', 'gris']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {DEVICE}")

In [ ]:
class TACOPascalVOCDataset(Dataset):
    def __init__(self, root_dir, split='train', transforms=None):
        self.root_dir = Path(root_dir) / split
        self.transforms = transforms
        self.imgs = sorted(list((self.root_dir / 'images').glob('*.jpg')))
        
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        label_path = self.root_dir / 'labels' / f"{img_path.stem}.xml"
        img = np.array(Image.open(img_path).convert("RGB"))
        
        tree = ET.parse(label_path)
        root = tree.getroot()
        
        boxes = []
        labels = []
        for obj in root.findall('object'):
            name = obj.find('name').text
            bndbox = obj.find('bndbox')
            xmin = float(bndbox.find('xmin').text)
            ymin = float(bndbox.find('ymin').text)
            xmax = float(bndbox.find('xmax').text)
            ymax = float(bndbox.find('ymax').text)
            
            if xmax - xmin <= 1.0 or ymax - ymin <= 1.0: continue
            
            boxes.append([max(0, xmin), max(0, ymin), min(img.shape[1], xmax), min(img.shape[0], ymax)])
            labels.append(CLASS_TO_IDX[name])
            
        boxes = np.array(boxes, dtype=np.float32)
        labels = np.array(labels, dtype=np.int64)
        
        if self.transforms:
            transformed = self.transforms(image=img, bboxes=boxes, class_labels=labels)
            img = transformed['image']
            boxes = torch.as_tensor(transformed['bboxes'], dtype=torch.float32)
            labels = torch.as_tensor(transformed['class_labels'], dtype=torch.int64)
        else:
            img = torch.as_tensor(img, dtype=torch.float32)
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
        target = {"boxes": boxes, "labels": labels}
        return img, target

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

def collate_fn(batch): return tuple(zip(*batch))

train_dataset = TACOPascalVOCDataset(SSD_DIR, split='train', transforms=get_train_transforms())
val_dataset = TACOPascalVOCDataset(SSD_DIR, split='val', transforms=get_val_transforms())
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, collate_fn=collate_fn)

In [ ]:
def create_ssd_model(num_classes):
    model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)
    in_channels = [c.in_channels for c in model.head.classification_head.module_list]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = torchvision.models.detection.ssd.SSDClassificationHead(in_channels, num_anchors, num_classes)
    return model

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, scaler):
    model.train()
    total_loss = 0
    for images, targets in tqdm(data_loader, desc="Training"):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        with autocast():
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += losses.item()
    return total_loss / len(data_loader)

@torch.no_grad()
def evaluate_loss(model, data_loader, device):
    model.train()
    total_loss = 0
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
    return total_loss / len(data_loader)

### Hyperparameter Tuning

In [ ]:
learning_rates = [1e-2, 1e-3, 5e-4]
weight_decays = [5e-4, 1e-4]
epochs_per_config = 10
early_stopping_patience = 3

best_global_val_loss = float('inf')
best_params = {}

for lr in learning_rates:
    for wd in weight_decays:
        print(f"\n>>> Testing: LR={lr}, WD={wd}")
        model = create_ssd_model(NUM_CLASSES).to(DEVICE)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
        scheduler = MultiStepLR(optimizer, milestones=[5, 8], gamma=0.1)
        scaler = GradScaler()
        
        config_best_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(epochs_per_config):
            t_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, scaler)
            v_loss = evaluate_loss(model, val_loader, DEVICE)
            scheduler.step()
            print(f"  Ep {epoch+1}: Train={t_loss:.4f}, Val={v_loss:.4f}, LR={scheduler.get_last_lr()[0]:.6f}")
            
            if v_loss < config_best_loss:
                config_best_loss = v_loss
                patience_counter = 0
                if v_loss < best_global_val_loss:
                    best_global_val_loss = v_loss
                    best_params = {'lr': lr, 'wd': wd}
            else:
                patience_counter += 1
                if patience_counter >= early_stopping_patience:
                    print("  Early stopping config...")
                    break

print(f"\nFINISH TUNING. Best Params: {best_params} (Loss: {best_global_val_loss:.4f})")

### Final Long Training

In [ ]:
FINAL_EPOCHS = 50
lr = best_params.get('lr', 1e-3)
wd = best_params.get('wd', 5e-4)

print(f"Starting Final Training with LR={lr}, WD={wd}...")
model = create_ssd_model(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)

scheduler = MultiStepLR(optimizer, milestones=[int(FINAL_EPOCHS*0.6), int(FINAL_EPOCHS*0.8)], gamma=0.1)
scaler = GradScaler()

best_val_loss = float('inf')

for epoch in range(FINAL_EPOCHS):
    t_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, scaler)
    v_loss = evaluate_loss(model, val_loader, DEVICE)
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{FINAL_EPOCHS} - Loss: {t_loss:.4f}, Val: {v_loss:.4f}")
    
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        save_path = PROJECT_DIR / 'ssd_implementation' / 'ssd300_best_model.pth'
        torch.save(model.state_dict(), save_path)
        print(f"  *** New Best Model Saved! Loss: {v_loss:.4f} ***")